# 40 — STA/LTA detection → association → windows → ObsPy pickers (baseline)

Outputs:
- CSV table of associated detections
- miniSEED windows per detection
- QuakeML catalog with baseline picks

Note: This is a baseline; tune STA/LTA + picker params for Redoubt signal types.

In [ ]:
%run 00_config.ipynb
from obspy import read
from obspy.signal.trigger import coincidence_trigger, pk_baer
from obspy.core.event import Catalog, Event, Origin, Pick, WaveformStreamID
from obspy.core.event.base import ResourceIdentifier
import glob

In [ ]:
wf_dir = os.path.join(ROOT, "waveforms_daily")
assert os.path.isdir(wf_dir), "Run 20_waveforms_daily.ipynb first"

det_dir = ensure_dir(os.path.join(ROOT, "detections"))
win_dir = ensure_dir(os.path.join(ROOT, "event_windows_mseed"))


In [ ]:
def associate_events_sta_lta(
    st_day,
    sta=1.0, lta=10.0,
    thr_on=3.5, thr_off=1.0,
    coincidence_sum=3,
    max_trigger_length=60,
    delete_long_trigger=True,
    trigger_off_extension=5.0,
    details=True,
):
    return coincidence_trigger(
        "recstalta", thr_on, thr_off, st_day,
        sta=sta, lta=lta,
        coincidence_sum=coincidence_sum,
        max_trigger_length=max_trigger_length,
        delete_long_trigger=delete_long_trigger,
        trigger_off_extension=trigger_off_extension,
        details=details
    )

def extract_event_window(st_day, t_event, pre=20.0, post=80.0, pad=True, fill_value=np.nan):
    t_start = t_event - pre
    t_end = t_event + post
    st_win = st_day.copy().trim(t_start, t_end, pad=pad, fill_value=fill_value)
    st_win = Stream([tr for tr in st_win if tr.stats.npts > 1])
    return st_win

def pick_pk_baer(tr, t_event, search_pre=10.0, search_post=20.0):
    tr2 = tr.copy().trim(t_event - search_pre, t_event + search_post, pad=True, fill_value=0.0)
    tr2.detrend("demean")
    tr2.taper(0.01)

    df = tr2.stats.sampling_rate
    p_pick_sample, _ = pk_baer(
        tr2.data.astype(np.float64),
        df,
        20,  # tdownmax
        60,  # tupevent
        7,   # thr1
        12,  # thr2
        100, # preset_len
        200  # p_dur
    )
    if p_pick_sample is None or p_pick_sample < 0:
        return None
    return tr2.stats.starttime + (p_pick_sample / df)

def event_tag(t_event):
    return t_event.strftime("%Y%m%dT%H%M%S")


In [ ]:
# Parameters (baseline starting point)
STA_S = 1.0
LTA_S = 10.0
THR_ON = 3.5
THR_OFF = 1.0
COINC_SUM = 3

PRE_S = 20.0
POST_S = 80.0

print("Params:",
      f"STA={STA_S}s LTA={LTA_S}s on={THR_ON} off={THR_OFF} coincidence={COINC_SUM}",
      f"window=({-PRE_S}, +{POST_S})")


In [ ]:
# Loop over days, associate, write outputs
records = []
cat_out = Catalog()

day_files = sorted(glob.glob(os.path.join(wf_dir, "*.Z.mseed")))
if not day_files:
    raise RuntimeError("No daily miniSEED files found.")

def day_from_fname(path):
    base = os.path.basename(path)
    parts = base.split(".")
    return parts[2]  # YYYY-MM-DD

days = sorted(set(day_from_fname(p) for p in day_files))
print("Days found:", days)

for day_str in days:
    det_csv = os.path.join(det_dir, f"detections_{day_str}.csv")
    det_quakeml = os.path.join(det_dir, f"detections_{day_str}.xml")

    if file_exists(det_csv) and file_exists(det_quakeml):
        print("Detections already exist for", day_str, "- skipping.")
        continue

    st_day = Stream()
    for p in day_files:
        if day_from_fname(p) == day_str:
            st_day += read(p)

    st_day.sort(keys=["network","station","location","channel","starttime"])

    print("Associating events for", day_str, "traces:", len(st_day))
    triglist = associate_events_sta_lta(
        st_day,
        sta=STA_S, lta=LTA_S,
        thr_on=THR_ON, thr_off=THR_OFF,
        coincidence_sum=COINC_SUM,
        details=True
    )

    print("Associated detections:", len(triglist))

    cat_day = Catalog()

    for i, evt in enumerate(triglist):
        t_event = evt["time"]
        stations = evt.get("stations", [])
        trig_dur = evt.get("duration", np.nan)

        st_win = extract_event_window(st_day, t_event, pre=PRE_S, post=POST_S)
        win_name = os.path.join(win_dir, f"EVT_{day_str}_{event_tag(t_event)}.mseed")
        st_win.write(win_name, format="MSEED")

        ev = Event(resource_id=ResourceIdentifier(f"smi:local/redoubt/{day_str}/evt{i:05d}"))
        ev.origins = [Origin(time=t_event)]

        for tr in st_day:
            if not tr.id.endswith("Z"):
                continue
            if tr.stats.station not in stations:
                continue
            ptime = pick_pk_baer(tr, t_event)
            if ptime is None:
                continue
            wid = WaveformStreamID(
                network_code=tr.stats.network,
                station_code=tr.stats.station,
                location_code=tr.stats.location,
                channel_code=tr.stats.channel
            )
            ev.picks.append(Pick(time=ptime, phase_hint="P", waveform_id=wid))

        cat_day.events.append(ev)

        records.append(dict(
            day=day_str,
            idx=i,
            time=t_event.datetime,
            duration=float(trig_dur) if trig_dur is not None else np.nan,
            nsta=len(stations),
            stations=",".join(stations),
            window_mseed=os.path.basename(win_name),
            npicks=len(ev.picks),
        ))

    df_det = pd.DataFrame(records).query("day == @day_str").sort_values("time")
    df_det.to_csv(det_csv, index=False)
    cat_day.write(det_quakeml, format="QUAKEML")

    print("Wrote:", det_csv)
    print("Wrote:", det_quakeml)

    cat_out += cat_day

all_quakeml = os.path.join(ROOT, "detections_all_days_with_picks.xml")
cat_out.write(all_quakeml, format="QUAKEML")
print("Wrote combined QuakeML:", all_quakeml)


In [ ]:
# Quick look at combined detection table
df_all = pd.DataFrame(records).sort_values("time")
display(df_all.head())
print("Total detections:", len(df_all))
